In [1]:
# Install Ultralytics YOLO
!pip install ultralytics -q
import os
import cv2
import shutil
import xml.etree.ElementTree as ET
from collections import defaultdict
from ultralytics import YOLO

print("Installation complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.4 MB/s eta 0:00:00a 0:00:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Installation complete.


In [2]:
# --- USER CONFIGURATION ---
# UPDATE THESE PATHS IF NECESSARY
# Assuming the video and XML have the same basename as provided in your prompt
VIDEO_PATH = "/kaggle/input/bdd-teslaped/NO20251114-155237-143770F.MP4" 
XML_PATH   = "/kaggle/input/bdd-teslaped/NO20251114-155237-143770F.xml"

# Output directories (Kaggle 'working' directory)
BASE_DIR = "/kaggle/working/dataset"
IMG_DIR = os.path.join(BASE_DIR, "images/train")
LBL_DIR = os.path.join(BASE_DIR, "labels/train")

# Create directories
os.makedirs(IMG_DIR, exist_ok=True)
os.makedirs(LBL_DIR, exist_ok=True)

print(f"Video Path set to: {VIDEO_PATH}")
print(f"XML Path set to: {XML_PATH}")

Video Path set to: /kaggle/input/bdd-teslaped/NO20251114-155237-143770F.MP4
XML Path set to: /kaggle/input/bdd-teslaped/NO20251114-155237-143770F.xml


In [3]:
cap = cv2.VideoCapture(VIDEO_PATH)

# Get video dimensions dynamically (More robust than XML metadata)
VID_WIDTH  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
VID_HEIGHT = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

frame_count = 0
saved_count = 0

print(f"Video Resolution: {VID_WIDTH}x{VID_HEIGHT}")
print(f"Total Frames: {total_frames}")
print("Starting frame extraction...")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Save every frame
    filename = f"frame_{frame_count:06d}.jpg"
    out_path = os.path.join(IMG_DIR, filename)
    cv2.imwrite(out_path, frame)
    
    saved_count += 1
    frame_count += 1
    
    if frame_count % 500 == 0:
        print(f"Extracted {frame_count} frames...", end='\r')

cap.release()
print(f"\nDone! Saved {saved_count} images to {IMG_DIR}")

Video Resolution: 2592x1944
Total Frames: 1800
Starting frame extraction...
Extracted 1500 frames...
Done! Saved 1800 images to /kaggle/working/dataset/images/train


In [4]:
# MAPPING CONFIGURATION
# Defined based on your CSV and XML Data
CLASSES = [
    "person",       # 0 (From 'pedestrian')
    "car",          # 1 (private-car, microbus, van)
    "rickshaw",     # 2 (paddle-rickshaw)
    "cng",          # 3
    "bus",          # 4
    "truck",        # 5
    "motorbike",    # 6 (bike)
    "bicycle",      # 7
    "leguna",       # 8
    "bangla-tesla"  # 9
]

# Map XML attributes (V_Type) to YOLO Class Names
TYPE_MAPPING = {
    "pedestrian": "person",
    "private-car": "car",
    "microbus": "car",
    "van": "car",
    "paddle-rickshaw": "rickshaw",
    "cng": "cng",
    "bus": "bus",
    "truck": "truck",
    "bike": "motorbike",
    "bicycle": "bicycle",
    "leguna": "leguna",
    "bangla-tesla": "bangla-tesla"
}

def parse_cvat_video_xml(xml_file, width, height):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    frame_data = defaultdict(list)

    # Loop through tracks
    for track in root.findall("track"):
        label = track.get("label")
        
        # Determine the class name based on label and attributes
        class_name = None
        
        # Case 1: Pedestrian
        if label == "pedestrian":
            class_name = TYPE_MAPPING.get("pedestrian")
            
        # Case 2: Vehicle (Requires checking V_Type attribute)
        elif label == "vehicle":
            v_type = None
            # Find the V_Type attribute inside the track (if static) or box (if dynamic)
            # Checking inside box usually covers both cases in CVAT 1.1
            first_box = track.find("box")
            if first_box is not None:
                 for attr in first_box.findall("attribute"):
                    if attr.get("name") == "V_Type":
                        v_type = attr.text
                        break
            
            # If not found in box, check track-level attributes (sometimes happens)
            if not v_type:
                 for attr in track.findall("attribute"):
                    if attr.get("name") == "V_Type":
                        v_type = attr.text
                        break

            if v_type and v_type in TYPE_MAPPING:
                class_name = TYPE_MAPPING[v_type]
        
        # Skip if class is not in our desired training list
        if class_name not in CLASSES:
            continue

        cls_id = CLASSES.index(class_name)

        # Iterate over all boxes in this track
        for box in track.findall("box"):
            frame_id = int(box.get("frame"))
            
            # Check for occlusion attribute if you want to filter (Optional)
            # occluded = box.get("occluded") == "1"

            # Coordinates
            xtl = float(box.get("xtl"))
            ytl = float(box.get("ytl"))
            xbr = float(box.get("xbr"))
            ybr = float(box.get("ybr"))

            # Normalize for YOLO format (center_x, center_y, width, height)
            box_width = xbr - xtl
            box_height = ybr - ytl
            x_center = xtl + (box_width / 2)
            y_center = ytl + (box_height / 2)

            x_n = x_center / width
            y_n = y_center / height
            w_n = box_width / width
            h_n = box_height / height

            # Safety clamp to [0, 1]
            x_n = max(0.0, min(1.0, x_n))
            y_n = max(0.0, min(1.0, y_n))
            w_n = max(0.0, min(1.0, w_n))
            h_n = max(0.0, min(1.0, h_n))

            line = f"{cls_id} {x_n:.6f} {y_n:.6f} {w_n:.6f} {h_n:.6f}"
            frame_data[frame_id].append(line)

    return frame_data

# Run Conversion
print("Parsing XML...")
# We pass the dimensions we read in Cell 3 to ensure accuracy
frames_map = parse_cvat_video_xml(XML_PATH, VID_WIDTH, VID_HEIGHT)

print(f"Generating labels for {len(frames_map)} frames...")
for frame_id, lines in frames_map.items():
    txt_filename = f"frame_{frame_id:06d}.txt"
    with open(os.path.join(LBL_DIR, txt_filename), "w") as f:
        f.write("\n".join(lines))

print("Labels generated successfully.")

Parsing XML...
Generating labels for 1800 frames...
Labels generated successfully.


In [5]:
yaml_content = f"""
path: {BASE_DIR}
train: images/train
val: images/train  # Using train as val for single-video training

nc: {len(CLASSES)}
names: {CLASSES}
"""

with open("/kaggle/working/data.yaml", "w") as f:
    f.write(yaml_content)

print("data.yaml created!")

data.yaml created!


In [6]:
# Load a model
model = YOLO('yolov8n.pt')  # Nano model (fastest)

# Train
results = model.train(
    data='/kaggle/working/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    project='/kaggle/working/runs',
    name='dashcam_teacher'
)

Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=dashcam_teacher, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plot

In [7]:
import shutil
import os
from IPython.display import FileLink

# --- CONFIGURATION ---
target_folder = "/kaggle/working/runs/dashcam_teacher"
output_zip_name = "yolo_training_results"

# --- EXECUTION ---
if os.path.exists(target_folder):
    print(f"Found training results in: {target_folder}")
    print("Zipping files... please wait.")
    
    shutil.make_archive(output_zip_name, 'zip', target_folder)
    
    print("\nSUCCESS! Click the link below to download your model and graphs:")
    display(FileLink(f"{output_zip_name}.zip"))
    
else:
    print(f"ERROR: Could not find folder '{target_folder}'")
    print("Please make sure Cell 6 finished training successfully.")

Found training results in: /kaggle/working/runs/dashcam_teacher
Zipping files... please wait.

SUCCESS! Click the link below to download your model and graphs:


/kaggle/working/yolo_training_results.zip